Bagging -- Random Forest

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

DataSet Load

In [2]:
df=pd.read_csv("Travel.csv")

In [3]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

Removing unwanted columns

In [5]:
## drop Customer ID column as it is not required for analysis
df.drop('CustomerID',axis=1,inplace=True)

Check categories

In [6]:
df['Gender'].value_counts()

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [7]:
df['Gender']=df['Gender'].replace("Fe Male","Female")

In [8]:
df["MaritalStatus"].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [9]:
df['MaritalStatus']=df['MaritalStatus'].replace("Single","Unmarried")

Check missing values

In [10]:
## Check Misssing Values
##these are the features with nan value
features_with_na=[features for features in df.columns if df[features].isnull().sum()>=1]
for feature in features_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,5), '% missing values')

Age 4.62357 % missing values
TypeofContact 0.51146 % missing values
DurationOfPitch 5.13502 % missing values
NumberOfFollowups 0.92062 % missing values
PreferredPropertyStar 0.53191 % missing values
NumberOfTrips 2.86416 % missing values
NumberOfChildrenVisiting 1.35025 % missing values
MonthlyIncome 4.76678 % missing values


Imputing Null values

    Impute Median value for Age column
    Impute Mode for Type of Contract
    Impute Median for Duration of Pitch
    Impute Mode for NumberofFollowup as it is Discrete feature
    Impute Mode for PreferredPropertyStar
    Impute Median for NumberofTrips
    Impute Mode for NumberOfChildrenVisiting
    Impute Median for MonthlyIncome
 

 first do EDA later decide imputing technique

Train test split

In [13]:
from sklearn.model_selection import train_test_split
X=df.drop('ProdTaken',axis=1)
y=df['ProdTaken']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=0)

In [22]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       4662 non-null   float64
 1   TypeofContact             4863 non-null   str    
 2   CityTier                  4888 non-null   int64  
 3   DurationOfPitch           4637 non-null   float64
 4   Occupation                4888 non-null   str    
 5   Gender                    4888 non-null   str    
 6   NumberOfPersonVisiting    4888 non-null   int64  
 7   NumberOfFollowups         4843 non-null   float64
 8   ProductPitched            4888 non-null   str    
 9   PreferredPropertyStar     4862 non-null   float64
 10  MaritalStatus             4888 non-null   str    
 11  NumberOfTrips             4748 non-null   float64
 12  Passport                  4888 non-null   int64  
 13  PitchSatisfactionScore    4888 non-null   int64  
 14  OwnCar             

In [ ]:
# # =========================
# # Missing Value Imputation
# # =========================

# # Age
# X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
# X_test['Age'] = X_test['Age'].fillna(X_train['Age'].median())

# # TypeofContact
# X_train['TypeofContact'] = X_train['TypeofContact'].fillna(
#     X_train['TypeofContact'].mode()[0]
# )
# X_test['TypeofContact'] = X_test['TypeofContact'].fillna(
#     X_train['TypeofContact'].mode()[0]
# )

# # DurationOfPitch
# X_train['DurationOfPitch'] = X_train['DurationOfPitch'].fillna(
#     X_train['DurationOfPitch'].median()
# )
# X_test['DurationOfPitch'] = X_test['DurationOfPitch'].fillna(
#     X_train['DurationOfPitch'].median()
# )

# # NumberOfFollowups
# X_train['NumberOfFollowups'] = X_train['NumberOfFollowups'].fillna(
#     X_train['NumberOfFollowups'].mode()[0]
# )
# X_test['NumberOfFollowups'] = X_test['NumberOfFollowups'].fillna(
#     X_train['NumberOfFollowups'].mode()[0]
# )

# # PreferredPropertyStar
# X_train['PreferredPropertyStar'] = X_train['PreferredPropertyStar'].fillna(
#     X_train['PreferredPropertyStar'].mode()[0]
# )
# X_test['PreferredPropertyStar'] = X_test['PreferredPropertyStar'].fillna(
#     X_train['PreferredPropertyStar'].mode()[0]
# )

# # NumberOfTrips
# X_train['NumberOfTrips'] = X_train['NumberOfTrips'].fillna(
#     X_train['NumberOfTrips'].median()
# )
# X_test['NumberOfTrips'] = X_test['NumberOfTrips'].fillna(
#     X_train['NumberOfTrips'].median()
# )

# # NumberOfChildrenVisiting
# X_train['NumberOfChildrenVisiting'] = X_train['NumberOfChildrenVisiting'].fillna(
#     X_train['NumberOfChildrenVisiting'].mode()[0]
# )
# X_test['NumberOfChildrenVisiting'] = X_test['NumberOfChildrenVisiting'].fillna(
#     X_train['NumberOfChildrenVisiting'].mode()[0]
# )

# # MonthlyIncome
# X_train['MonthlyIncome'] = X_train['MonthlyIncome'].fillna(
#     X_train['MonthlyIncome'].median()
# )
# X_test['MonthlyIncome'] = X_test['MonthlyIncome'].fillna(
#     X_train['MonthlyIncome'].median()
# )

Even better: use SimpleImputer

In a real ML project, you would normally use sklearn's SimpleImputer inside a preprocessing pipeline. This makes the process cleaner and automatically ensures the imputation statistics are learned from training data only.

In [14]:
from sklearn.impute import SimpleImputer

# Numerical columns
num_cols = [
    'Age',
    'DurationOfPitch',
    'NumberOfTrips',
    'MonthlyIncome'
]

# Categorical/discrete columns
cat_cols = [
    'TypeofContact',
    'NumberOfFollowups',
    'PreferredPropertyStar',
    'NumberOfChildrenVisiting'
]

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

In [15]:
# Create new feature in X_train
X_train['TotalVisiting'] = (
    X_train['NumberOfPersonVisiting'] +
    X_train['NumberOfChildrenVisiting']
)

# Create new feature in X_test
X_test['TotalVisiting'] = (
    X_test['NumberOfPersonVisiting'] +
    X_test['NumberOfChildrenVisiting']
)

# Drop the original columns from X_train
X_train.drop(
    columns=['NumberOfPersonVisiting', 'NumberOfChildrenVisiting'],
    inplace=True
)

# Drop the original columns from X_test
X_test.drop(
    columns=['NumberOfPersonVisiting', 'NumberOfChildrenVisiting'],
    inplace=True
)

Feature Extraction or Scaling

In [20]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 3421 entries, 118 to 2732
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     3421 non-null   float64
 1   TypeofContact           3421 non-null   str    
 2   CityTier                3421 non-null   int64  
 3   DurationOfPitch         3421 non-null   float64
 4   Occupation              3421 non-null   str    
 5   Gender                  3421 non-null   str    
 6   NumberOfFollowups       3421 non-null   object 
 7   ProductPitched          3421 non-null   str    
 8   PreferredPropertyStar   3421 non-null   object 
 9   MaritalStatus           3421 non-null   str    
 10  NumberOfTrips           3421 non-null   float64
 11  Passport                3421 non-null   int64  
 12  PitchSatisfactionScore  3421 non-null   int64  
 13  OwnCar                  3421 non-null   int64  
 14  Designation             3421 non-null   str    
 15  M

In [23]:
# Columns that should be numeric
numeric_conversion_cols = [
    'NumberOfFollowups',
    'PreferredPropertyStar',
    'TotalVisiting'
]

# Convert X_train
X_train[numeric_conversion_cols] = X_train[numeric_conversion_cols].apply(
    pd.to_numeric, errors='coerce'
)

# Convert X_test
X_test[numeric_conversion_cols] = X_test[numeric_conversion_cols].apply(
    pd.to_numeric, errors='coerce'
)

In [24]:
X_train.head()

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisiting
118,36.0,Self Enquiry,1,9.0,Salaried,Female,3.0,Basic,3.0,Married,6.0,0,2,1,Executive,17835.0,2.0
261,38.0,Company Invited,3,8.0,Salaried,Male,4.0,Deluxe,3.0,Divorced,4.0,0,5,1,Manager,20249.0,3.0
598,28.0,Self Enquiry,1,13.0,Small Business,Male,3.0,Basic,3.0,Unmarried,7.0,0,3,0,Executive,22338.0,2.0
2154,40.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Standard,4.0,Married,3.0,0,4,0,Senior Manager,24705.0,3.0
905,29.0,Self Enquiry,1,6.0,Salaried,Female,4.0,Super Deluxe,3.0,Married,4.0,0,2,0,AVP,31124.0,3.0


In [25]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 3421 entries, 118 to 2732
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     3421 non-null   float64
 1   TypeofContact           3421 non-null   str    
 2   CityTier                3421 non-null   int64  
 3   DurationOfPitch         3421 non-null   float64
 4   Occupation              3421 non-null   str    
 5   Gender                  3421 non-null   str    
 6   NumberOfFollowups       3421 non-null   float64
 7   ProductPitched          3421 non-null   str    
 8   PreferredPropertyStar   3421 non-null   float64
 9   MaritalStatus           3421 non-null   str    
 10  NumberOfTrips           3421 non-null   float64
 11  Passport                3421 non-null   int64  
 12  PitchSatisfactionScore  3421 non-null   int64  
 13  OwnCar                  3421 non-null   int64  
 14  Designation             3421 non-null   str    
 15  M

In [26]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Get categorical and numerical features from X_train
cat_features = X_train.select_dtypes(include="object").columns
num_features = X_train.select_dtypes(exclude="object").columns

# Transformers
numeric_transformer = StandardScaler()

oh_transformer = OneHotEncoder(
    drop='first',
    handle_unknown='ignore'
)

# Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

C:\Users\dheer\AppData\Local\Temp\ipykernel_12116\1584144077.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train.select_dtypes(include="object").columns


In [27]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('OneHotEncoder', ...), ('StandardScaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name``

In [28]:
X_train=preprocessor.fit_transform(X_train)

In [29]:
X_test=preprocessor.transform(X_test)

X_train and X_test is ready...

Bagging

In [30]:
from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay,precision_score, recall_score, f1_score, roc_auc_score,roc_curve 

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


Comparing models

In [33]:
models={
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier()
}

In [34]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Training set performance
    model_train_accuracy = accuracy_score(y_train, y_train_pred) # Calculate Accuracy
    model_train_f1 = f1_score(y_train, y_train_pred, average='weighted') # Calculate F1-score
    model_train_precision = precision_score(y_train, y_train_pred) # Calculate Precision
    model_train_recall = recall_score(y_train, y_train_pred) # Calculate Recall
    model_train_rocauc_score = roc_auc_score(y_train, y_train_pred)


    # Test set performance
    model_test_accuracy = accuracy_score(y_test, y_test_pred) # Calculate Accuracy
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted') # Calculate F1-score
    model_test_precision = precision_score(y_test, y_test_pred) # Calculate Precision
    model_test_recall = recall_score(y_test, y_test_pred) # Calculate Recall
    model_test_rocauc_score = roc_auc_score(y_test, y_test_pred) #Calculate Roc


    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(model_train_accuracy))
    print('- F1 score: {:.4f}'.format(model_train_f1))
    
    print('- Precision: {:.4f}'.format(model_train_precision))
    print('- Recall: {:.4f}'.format(model_train_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

    
    
    print('----------------------------------')
    
    print('Model performance for Test set')
    print('- Accuracy: {:.4f}'.format(model_test_accuracy))
    print('- F1 score: {:.4f}'.format(model_test_f1))
    print('- Precision: {:.4f}'.format(model_test_precision))
    print('- Recall: {:.4f}'.format(model_test_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    
    print('='*35)
    print('\n')

Logistic Regression
Model performance for Training set
- Accuracy: 0.8518
- F1 score: 0.8296
- Precision: 0.7296
- Recall: 0.3457
- Roc Auc Score: 0.6579
----------------------------------
Model performance for Test set
- Accuracy: 0.8371
- F1 score: 0.8022
- Precision: 0.6813
- Recall: 0.2279
- Roc Auc Score: 0.6018


Decision Tree
Model performance for Training set
- Accuracy: 1.0000
- F1 score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- Roc Auc Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.9202
- F1 score: 0.9192
- Precision: 0.8039
- Recall: 0.7537
- Roc Auc Score: 0.8559


RandomForest
Model performance for Training set
- Accuracy: 1.0000
- F1 score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- Roc Auc Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.9107
- F1 score: 0.9010
- Precision: 0.9434
- Recall: 0.5515
- Roc Auc Score: 0.7720




Bagging

M1   → Logistic Regression

M2   → Decision Tree

M3   → Logistic Regression 

M4   → Decision Tree

...

M100 → Decision Tree

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
import numpy as np

models = []

for i in range(100):

    if i % 2 == 0:
        model = LogisticRegression(max_iter=1000)
    else:
        model = DecisionTreeClassifier(random_state=42)

    models.append(model)

In [38]:
trained_models = []

n_samples = X_train.shape[0]

for i, model in enumerate(models):

    # Randomly select training rows WITH replacement
    indices = np.random.choice(
        n_samples,
        size=n_samples,
        replace=True
    )

    X_sample = X_train[indices]
    y_sample = y_train.iloc[indices]

    # Train the model on its own bootstrap sample
    model.fit(X_sample, y_sample)

    trained_models.append(model)

In [40]:
predictions = []

for model in trained_models:
    pred = model.predict(X_test)
    predictions.append(pred)

predictions = np.array(predictions)

In [41]:
predictions

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0]], shape=(100, 1467))

In [42]:
y_pred = (predictions.mean(axis=0) >= 0.5).astype(int)

In [44]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.98      0.92      1195
           1       0.79      0.31      0.44       272

    accuracy                           0.86      1467
   macro avg       0.83      0.64      0.68      1467
weighted avg       0.85      0.86      0.83      1467



Random Forest

In [45]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(n_estimators=100, random_state=42) # n_estimators means number of trees in the forest
rf.fit(X_train, y_train) # Fit the model on training data
y_pred=rf.predict(X_test) # Predict on the test data
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.99      0.95      1195
           1       0.96      0.55      0.70       272

    accuracy                           0.91      1467
   macro avg       0.93      0.77      0.82      1467
weighted avg       0.92      0.91      0.90      1467



Hyperparameter tuning

In [46]:
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

In [47]:
# Models list for Hyperparameter tuning
randomcv_models = [
                   ("RF", RandomForestClassifier(), rf_params)
                   
                   ]

In [48]:
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits


c:\Users\dheer\anaconda3\envs\deeplearning\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
81 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
42 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\dheer\anaconda3\envs\deeplearning\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\dheer\anaconda3\envs\deeplearning\Lib\site-packages\sklearn\base.py", line 1393, in wrapper
    estimator._validate_params()
  File "c:\Users\dheer\anaconda3\envs\deeplearning\Lib\site-packages\sklearn\base.py", line 554, in _validate_params
    validate_p

---------------- Best Params for RF -------------------
{'n_estimators': 500, 'min_samples_split': 2, 'max_features': 8, 'max_depth': 15}


for the above best params we are going to train

In [49]:
models={
    
    "Random Forest":RandomForestClassifier(n_estimators=500,min_samples_split=2,
                                          max_features=8,max_depth=15)
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Training set performance
    model_train_accuracy = accuracy_score(y_train, y_train_pred) # Calculate Accuracy
    model_train_f1 = f1_score(y_train, y_train_pred, average='weighted') # Calculate F1-score
    model_train_precision = precision_score(y_train, y_train_pred) # Calculate Precision
    model_train_recall = recall_score(y_train, y_train_pred) # Calculate Recall
    model_train_rocauc_score = roc_auc_score(y_train, y_train_pred)


    # Test set performance
    model_test_accuracy = accuracy_score(y_test, y_test_pred) # Calculate Accuracy
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted') # Calculate F1-score
    model_test_precision = precision_score(y_test, y_test_pred) # Calculate Precision
    model_test_recall = recall_score(y_test, y_test_pred) # Calculate Recall
    model_test_rocauc_score = roc_auc_score(y_test, y_test_pred) #Calculate Roc


    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(model_train_accuracy))
    print('- F1 score: {:.4f}'.format(model_train_f1))
    
    print('- Precision: {:.4f}'.format(model_train_precision))
    print('- Recall: {:.4f}'.format(model_train_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

    
    
    print('----------------------------------')
    
    print('Model performance for Test set')
    print('- Accuracy: {:.4f}'.format(model_test_accuracy))
    print('- F1 score: {:.4f}'.format(model_test_f1))
    print('- Precision: {:.4f}'.format(model_test_precision))
    print('- Recall: {:.4f}'.format(model_test_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    
    print('='*35)
    print('\n')

Random Forest
Model performance for Training set
- Accuracy: 0.9994
- F1 score: 0.9994
- Precision: 1.0000
- Recall: 0.9969
- Roc Auc Score: 0.9985
----------------------------------
Model performance for Test set
- Accuracy: 0.9189
- F1 score: 0.9115
- Precision: 0.9422
- Recall: 0.5993
- Roc Auc Score: 0.7954




Same way for AdaBoost,Gradient Boose,XgBoost

In [51]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.3/101.7 MB 6.7 MB/s eta 0:00:15
   - -------------------------------------- 2.6/101.7 MB 6.6 MB/s eta 0:00:16
   - -------------------------------------- 3.7/101.7 MB 5.9 MB/s eta 0:00:17
   - -------------------------------------- 5.0/101.7 MB 5.9 MB/s eta 0:00:17
   -- ------------------------------------- 6.3/101.7 MB 5.9 MB/s eta 0:00:17
   -- ------------------------------------- 7.3/101.7 MB 5.8 MB/s eta 0:00:17
   --- ------------------------------------ 8.4/101.7 MB 5.7 MB/s eta 0:00:17
   --- ------------------------------------ 9.4/101.7 MB 5.5 MB/s eta 0:00:17
   ---- ----------------------------------- 10.7/101.7 MB 5.5 MB/s eta 0:00:17
   ---- ----------------------------------- 11.8/101.7 MB 5.5 MB/s eta 0:00:17
   ----- ---------------------------------- 12.8/101.7 MB 5.4 MB/s eta 0:00:17
   ----- ---------------------------------- 14.2/101.7 MB 5.4 MB/s e

In [52]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

In [53]:
models={
    "Logisitic Regression":LogisticRegression(),
    "Decision Tree":DecisionTreeClassifier(),
    "Random Forest":RandomForestClassifier(),
    "Gradient Boost":GradientBoostingClassifier(),
    "Adaboost":AdaBoostClassifier(),
    "Xgboost":XGBClassifier()
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Training set performance
    model_train_accuracy = accuracy_score(y_train, y_train_pred) # Calculate Accuracy
    model_train_f1 = f1_score(y_train, y_train_pred, average='weighted') # Calculate F1-score
    model_train_precision = precision_score(y_train, y_train_pred) # Calculate Precision
    model_train_recall = recall_score(y_train, y_train_pred) # Calculate Recall
    model_train_rocauc_score = roc_auc_score(y_train, y_train_pred)


    # Test set performance
    model_test_accuracy = accuracy_score(y_test, y_test_pred) # Calculate Accuracy
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted') # Calculate F1-score
    model_test_precision = precision_score(y_test, y_test_pred) # Calculate Precision
    model_test_recall = recall_score(y_test, y_test_pred) # Calculate Recall
    model_test_rocauc_score = roc_auc_score(y_test, y_test_pred) #Calculate Roc


    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(model_train_accuracy))
    print('- F1 score: {:.4f}'.format(model_train_f1))
    
    print('- Precision: {:.4f}'.format(model_train_precision))
    print('- Recall: {:.4f}'.format(model_train_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

    
    
    print('----------------------------------')
    
    print('Model performance for Test set')
    print('- Accuracy: {:.4f}'.format(model_test_accuracy))
    print('- F1 score: {:.4f}'.format(model_test_f1))
    print('- Precision: {:.4f}'.format(model_test_precision))
    print('- Recall: {:.4f}'.format(model_test_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    
    print('='*35)
    print('\n')

Logisitic Regression
Model performance for Training set
- Accuracy: 0.8518
- F1 score: 0.8296
- Precision: 0.7296
- Recall: 0.3457
- Roc Auc Score: 0.6579
----------------------------------
Model performance for Test set
- Accuracy: 0.8371
- F1 score: 0.8022
- Precision: 0.6813
- Recall: 0.2279
- Roc Auc Score: 0.6018


Decision Tree
Model performance for Training set
- Accuracy: 1.0000
- F1 score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- Roc Auc Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.9182
- F1 score: 0.9170
- Precision: 0.8016
- Recall: 0.7426
- Roc Auc Score: 0.8504


Random Forest
Model performance for Training set
- Accuracy: 1.0000
- F1 score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- Roc Auc Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.9121
- F1 score: 0.9025
- Precision: 0.9497
- Recall: 0.5551
- Roc Auc Score: 0.7742


Gradient Boost
Model performance for Training se

Hyperparameter tuning

In [62]:
# =========================
# Random Forest
# =========================

rf_params = {
    "max_depth": [5, 8, 10, 15, None],
    "max_features": [5, 7, 8, "sqrt", "log2"],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, 1000]
}


# =========================
# AdaBoost
# =========================

adaboost_param = {
    "n_estimators": [50, 60, 70, 80, 90],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]
}


# =========================
# Gradient Boosting
# =========================

gradient_params = {
    "loss": ["log_loss", "exponential"],
    "criterion": ["friedman_mse", "squared_error"],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500],
    "max_depth": [5, 8, 10, 15, None]
}


# =========================
# XGBoost
# =========================

xgboost_params = {
    "learning_rate": [0.1, 0.01],
    "max_depth": [5, 8, 12, 20, 30],
    "n_estimators": [100, 200, 300],
    "colsample_bytree": [0.3, 0.4, 0.5, 0.8, 1.0]
}


In [63]:
# =========================
# Models
# =========================

randomcv_models = [
    ("RF", RandomForestClassifier(random_state=42), rf_params),

    ("Adaboost", AdaBoostClassifier(random_state=42),
     adaboost_param),

    ("Gradient Boost", GradientBoostingClassifier(random_state=42),
     gradient_params),

    ("Xgboost", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ), xgboost_params)
]


In [64]:
from sklearn.model_selection import RandomizedSearchCV
import warnings
warnings.filterwarnings("ignore")
# =========================
# Randomized Search
# =========================

model_param = {}

for name, model, params in randomcv_models:

    random = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=20,       # 20 random combinations
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42,
        scoring="accuracy"
    )

    random.fit(X_train, y_train)

    model_param[name] = random.best_params_


# =========================
# Best Parameters
# =========================

for model_name in model_param:

    print(
        f"---------------- Best Params for {model_name} -------------------"
    )

    print(model_param[model_name])

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
---------------- Best Params for RF -------------------
{'n_estimators': 200, 'min_samples_split': 2, 'max_features': 'log2', 'max_depth': None}
---------------- Best Params for Adaboost -------------------
{'n_estimators': 50, 'learning_rate': 1.0}
---------------- Best Params for Gradient Boost -------------------
{'n_estimators': 500, 'min_samples_split': 15, 'max_depth': 15, 'loss': 'exponential', 'criterion': 'friedman_mse'}
---------------- Best Params for Xgboost -------------------
{'n_estimators': 300, 'max_depth': 30, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


using best params for training and evaluating models

In [65]:
models={
    
    "Random Forest":RandomForestClassifier(n_estimators= 200, min_samples_split= 2, max_features='log2', max_depth= None),
    "AdaBoost":AdaBoostClassifier(n_estimators=50,learning_rate=1.0),
    "Gradient Boost":GradientBoostingClassifier(loss='exponential',criterion='friedman_mse',min_samples_split=15,n_estimators=500,max_depth=15),
    "Xgboost":XGBClassifier(n_estimators=300,max_depth=30,learning_rate=0.1,
                           colsample_bytree=0.8)
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Training set performance
    model_train_accuracy = accuracy_score(y_train, y_train_pred) # Calculate Accuracy
    model_train_f1 = f1_score(y_train, y_train_pred, average='weighted') # Calculate F1-score
    model_train_precision = precision_score(y_train, y_train_pred) # Calculate Precision
    model_train_recall = recall_score(y_train, y_train_pred) # Calculate Recall
    model_train_rocauc_score = roc_auc_score(y_train, y_train_pred)


    # Test set performance
    model_test_accuracy = accuracy_score(y_test, y_test_pred) # Calculate Accuracy
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted') # Calculate F1-score
    model_test_precision = precision_score(y_test, y_test_pred) # Calculate Precision
    model_test_recall = recall_score(y_test, y_test_pred) # Calculate Recall
    model_test_rocauc_score = roc_auc_score(y_test, y_test_pred) #Calculate Roc


    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(model_train_accuracy))
    print('- F1 score: {:.4f}'.format(model_train_f1))
    
    print('- Precision: {:.4f}'.format(model_train_precision))
    print('- Recall: {:.4f}'.format(model_train_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

    
    
    print('----------------------------------')
    
    print('Model performance for Test set')
    print('- Accuracy: {:.4f}'.format(model_test_accuracy))
    print('- F1 score: {:.4f}'.format(model_test_f1))
    print('- Precision: {:.4f}'.format(model_test_precision))
    print('- Recall: {:.4f}'.format(model_test_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    
    print('='*35)
    print('\n')

Random Forest
Model performance for Training set
- Accuracy: 1.0000
- F1 score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- Roc Auc Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.9080
- F1 score: 0.8975
- Precision: 0.9419
- Recall: 0.5368
- Roc Auc Score: 0.7646


AdaBoost
Model performance for Training set
- Accuracy: 0.8518
- F1 score: 0.8227
- Precision: 0.7950
- Recall: 0.2932
- Roc Auc Score: 0.6378
----------------------------------
Model performance for Test set
- Accuracy: 0.8344
- F1 score: 0.7908
- Precision: 0.7164
- Recall: 0.1765
- Roc Auc Score: 0.5803


Gradient Boost
Model performance for Training set
- Accuracy: 1.0000
- F1 score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- Roc Auc Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.9496
- F1 score: 0.9473
- Precision: 0.9541
- Recall: 0.7647
- Roc Auc Score: 0.8782


Xgboost
Model performance for Training set
- Accuracy: 1.00

Gradient is best